Compare text vectorization methods: One-Hot, BOW, TF-IDF, Word2Vec, Sentence-BERT.

In [1]:
import numpy as np
from sklearn.feature_extraction.text import CountVectorizer, TfidfVectorizer
from sklearn.metrics.pairwise import cosine_similarity

In [2]:
# Prepare the corpus for the tutorial: deliberately includes synonyms + unrelated sentence
corpus = [
    "the cat sat on the mat",
    "the dog sat on the rug",
    "the king ruled the kingdom",
    "stock prices rose sharply today",
]

print("=" * 70)
print("CORPUS")
print("=" * 70)
for i, doc in enumerate(corpus):
    print(f"D{i}: {doc}")

CORPUS
D0: the cat sat on the mat
D1: the dog sat on the rug
D2: the king ruled the kingdom
D3: stock prices rose sharply today


# 1. ONE-HOT ENCODING (word-level, manual)
Each word in a vocabulary gets a unique vector with a single 1 and the rest 0s.

In [3]:
vocab = sorted(set(" ".join(corpus).split()))
word2idx = {w: i for i, w in enumerate(vocab)}
print(f"Vocab size: {len(vocab)}") # To get the count of the unique vocabs in our corpus

Vocab size: 15


In [4]:
def one_hot(word):
    vec = np.zeros(len(vocab))
    vec[word2idx[word]] = 1
    return vec

In [5]:
cat_vec = one_hot("cat")
dog_vec = one_hot("dog")
king_vec = one_hot("king")

print(f"\nVector length: {len(cat_vec)} (sparse, mostly zeros)")
print(f"cosine(cat, dog)  = {cosine_similarity([cat_vec], [dog_vec])[0][0]:.3f}")
print(f"cosine(cat, king) = {cosine_similarity([cat_vec], [king_vec])[0][0]:.3f}")
print(">> One-hot treats ALL word pairs as equally unrelated (similarity = 0).")


Vector length: 15 (sparse, mostly zeros)
cosine(cat, dog)  = 0.000
cosine(cat, king) = 0.000
>> One-hot treats ALL word pairs as equally unrelated (similarity = 0).


Each word gets its own one-hot vector of length 15 (vocabulary size), with a single 1 in that word's position and 0 everywhere else. For example "cat" → 1 in the "cat" slot, "dog" → 1 in the "dog" slot — different slots entirely.

Because no two distinct words ever share a 1 in the same position, the dot product (and therefore cosine similarity) between any two different words' vectors is always exactly 0. That's why both `cosine(cat, dog)` and `cosine(cat, king)` come out as 0.000 — identically — even though intuitively "cat" and "dog" (both animals) should be more related than "cat" and "king".

This is the core limitation OHC illustrates: it encodes *identity* (which word is this) but carries zero information about *meaning or relatedness*. Every word is equally "far" from every other word, regardless of how semantically close they actually are. This is precisely the gap that word embeddings (Word2Vec, BERT, etc.) are designed to close.

# 2. BAG OF WORDS (document level)
Represents a document as a vector of word counts (or presence/absence), ignoring order and grammar.

In [6]:
bow_vectorizer = CountVectorizer()
bow_matrix = bow_vectorizer.fit_transform(corpus)
bow_array = bow_matrix.toarray()

print(f"Vocab size: {len(bow_vectorizer.get_feature_names_out())}")
print(f"Matrix shape: {bow_array.shape}  (docs x vocab)")

Vocab size: 15
Matrix shape: (4, 15)  (docs x vocab)


What does the Matrix shape mean?

Vocabulary (15 unique words across the corpus, alphabetically sorted by CountVectorizer):
```
cat, dog, kingdom, mat, on, prices, ruled, rose, rug, sat, sharply, stock, the, today, king
```
Each document becomes a vector of length 15 — one slot per vocabulary word, holding that word's count in the document. With 4 documents, the result is a 4×15 matrix: 4 rows (one per document), 15 columns (one per vocabulary word).

| | cat | dog | king | kingdom | mat | on | prices | rose | rug | ruled | sat | sharply | stock | the | today |
|---|---|---|---|---|---|---|---|---|---|---|---|---|---|---|---|
| D0 | 1 | 0 | 0 | 0 | 1 | 1 | 0 | 0 | 0 | 0 | 1 | 0 | 0 | 2 | 0 |
| D1 | 0 | 1 | 0 | 0 | 0 | 1 | 0 | 0 | 1 | 0 | 1 | 0 | 0 | 2 | 0 |
| D2 | 0 | 0 | 1 | 1 | 0 | 0 | 0 | 0 | 0 | 1 | 0 | 0 | 0 | 2 | 0 |
| D3 | 0 | 0 | 0 | 0 | 0 | 0 | 1 | 1 | 0 | 0 | 0 | 1 | 1 | 0 | 1 |


In [7]:
bow_sim = cosine_similarity(bow_array)
print("\nDocument similarity matrix (BOW):")
print(np.round(bow_sim, 2))
print(">> D0 (cat) and D1 (dog) are similar mainly because they SHARE common words like 'the', 'sat', 'on' -- not because cat~dog semantically.")


Document similarity matrix (BOW):
[[1.   0.75 0.53 0.  ]
 [0.75 1.   0.53 0.  ]
 [0.53 0.53 1.   0.  ]
 [0.   0.   0.   1.  ]]
>> D0 (cat) and D1 (dog) are similar mainly because they SHARE common words like 'the', 'sat', 'on' -- not because cat~dog semantically.


This is a 4×4 matrix of cosine similarity scores between every pair of documents (D0–D3), using their BOW vectors from the table above. Row/column order matches D0, D1, D2, D3.

| | D0 | D1 | D2 | D3 |
|---|---|---|---|---|
| **D0** | 1.00 | 0.75 | 0.53 | 0.00 |
| **D1** | 0.75 | 1.00 | 0.53 | 0.00 |
| **D2** | 0.53 | 0.53 | 1.00 | 0.00 |
| **D3** | 0.00 | 0.00 | 0.00 | 1.00 |

Reading it:

The diagonal is always 1.00 — every document is perfectly similar to itself.

D0 vs D1 (cat/mat vs dog/rug) = 0.75, the highest off-diagonal score. Looking at the cross-tab, both sentences share "the" (×2), "sat", and "on" — four overlapping word-slots out of relatively short vectors, so they look very similar to BOW even though "cat" and "dog" are different words and "mat" and "rug" are different words.

D0 vs D2 and D1 vs D2 = 0.53. D2 ("the king ruled the kingdom") shares only "the" (×2) with D0 and D1, so it's somewhat similar but less so than D0/D1 are to each other.

D3 vs everything = 0.00. "Stock prices rose sharply today" shares zero words with the other three sentences, so BOW says it's completely dissimilar.

The key point: BOW similarity is purely about shared word counts, with no understanding of meaning. D0 and D1 score highly mainly because they both repeat the stopword "the" twice and share "sat"/"on" — not because "cat" and "dog" are semantically related. This is exactly the weakness TF-IDF tries to fix (by down-weighting "the") and that word embeddings fix more fully (by capturing that "cat" and "dog" are conceptually similar even when the exact words differ).

# 3. TF-IDF

--> Refines BOW by weighting words by importance, not just raw count.

* TF: how often a word appears in a document

* IDF: how rare a word is across all documents — common words (the, is) get down-weighted, rare/distinctive words get up-weighted

In [8]:
tfidf_vectorizer = TfidfVectorizer()
tfidf_matrix = tfidf_vectorizer.fit_transform(corpus)
tfidf_array = tfidf_matrix.toarray()

tfidf_sim = cosine_similarity(tfidf_array)
print("Document similarity matrix (TF-IDF):")
print(np.round(tfidf_sim, 2))

Document similarity matrix (TF-IDF):
[[1.   0.59 0.34 0.  ]
 [0.59 1.   0.34 0.  ]
 [0.34 0.34 1.   0.  ]
 [0.   0.   0.   1.  ]]


Same structure as the BOW matrix — pairwise cosine similarity between D0–D3 — but now using TF-IDF weighted vectors instead of raw counts.

| | D0 | D1 | D2 | D3 |
|---|---|---|---|---|
| **D0** | 1.00 | 0.59 | 0.34 | 0.00 |
| **D1** | 0.59 | 1.00 | 0.34 | 0.00 |
| **D2** | 0.34 | 0.34 | 1.00 | 0.00 |
| **D3** | 0.00 | 0.00 | 0.00 | 1.00 |

Compare directly to the BOW matrix from before:

| Pair | BOW | TF-IDF |
|---|---|---|
| D0–D1 | 0.75 | 0.59 |
| D0–D2 / D1–D2 | 0.53 | 0.34 |
| D3 vs others | 0.00 | 0.00 |

Every similarity score dropped. The reason is "the" — it appears in D0, D1, and D2 twice each, so under BOW it contributes heavily to their similarity. TF-IDF down-weights "the" because it's common across documents (low IDF), so its contribution to similarity shrinks. What's left driving the remaining similarity is the genuinely shared, more distinctive words: "sat"/"on" for D0–D1, and just the residual "the" weight for pairs involving D2.

D3 stays at 0.00 with everyone, same as before — it shares no vocabulary at all with the other three documents, so weighting changes nothing there.

The teaching takeaway: TF-IDF doesn't change *which* documents are most similar (D0–D1 is still the highest pair), but it gives a more honest similarity score by suppressing the inflating effect of common stopwords like "the" and letting more meaningful, distinguishing words drive the comparison.

In [9]:
# Show how common word "the" gets down-weighted vs distinctive word.
# IMPORTANT: compare IDF directly (per-occurrence weight), not the final
# TF-IDF score -- the final score also depends on raw term frequency (TF),
# which can mask the down-weighting effect for words that repeat a lot.

feature_names = list(tfidf_vectorizer.get_feature_names_out())
the_idx = feature_names.index("the")
sat_idx = feature_names.index("sat")

idf_the = tfidf_vectorizer.idf_[the_idx]
idf_sat = tfidf_vectorizer.idf_[sat_idx]

tf_the_d0 = corpus[0].split().count("the")   # raw count of "the" in D0
tf_sat_d0 = corpus[0].split().count("sat")   # raw count of "sat" in D0

print(f"\nDoc frequency: 'the' appears in {sum(1 for d in corpus if 'the' in d.split())}/{len(corpus)} docs "
      f"-> IDF = {idf_the:.3f}")
print(f"Doc frequency: 'sat' appears in {sum(1 for d in corpus if 'sat' in d.split())}/{len(corpus)} docs "
      f"-> IDF = {idf_sat:.3f}")
print(">> 'the' is common across documents -> LOWER IDF (down-weighted PER OCCURRENCE).")
print("   'sat' is rarer across documents   -> HIGHER IDF (up-weighted PER OCCURRENCE).")

print(f"\nIn D0, raw count: 'the' x{tf_the_d0}, 'sat' x{tf_sat_d0}")
print(f"Final TF-IDF weight of 'the' in D0: {tfidf_array[0][the_idx]:.3f}  "
      f"(= TF {tf_the_d0} x IDF {idf_the:.3f}, then L2-normalized)")
print(f"Final TF-IDF weight of 'sat' in D0: {tfidf_array[0][sat_idx]:.3f}  "
      f"(= TF {tf_sat_d0} x IDF {idf_sat:.3f}, then L2-normalized)")
print(">> 'the' still ends up with a sizeable final weight here ONLY because it occurs")
print("   twice in D0 -- its raw frequency partly offsets its low IDF. The down-weighting")
print("   is clearest in the IDF values above, not the final TF-IDF number alone.")
print(">> Net effect at the document level: shared stopwords contribute less than in BOW,")
print("   so D0-D1 similarity drops vs BOW (driven more by genuinely shared words like 'sat'/'on').")


Doc frequency: 'the' appears in 3/4 docs -> IDF = 1.223
Doc frequency: 'sat' appears in 2/4 docs -> IDF = 1.511
>> 'the' is common across documents -> LOWER IDF (down-weighted PER OCCURRENCE).
   'sat' is rarer across documents   -> HIGHER IDF (up-weighted PER OCCURRENCE).

In D0, raw count: 'the' x2, 'sat' x1
Final TF-IDF weight of 'the' in D0: 0.578  (= TF 2 x IDF 1.223, then L2-normalized)
Final TF-IDF weight of 'sat' in D0: 0.357  (= TF 1 x IDF 1.511, then L2-normalized)
>> 'the' still ends up with a sizeable final weight here ONLY because it occurs
   twice in D0 -- its raw frequency partly offsets its low IDF. The down-weighting
   is clearest in the IDF values above, not the final TF-IDF number alone.
>> Net effect at the document level: shared stopwords contribute less than in BOW,
   so D0-D1 similarity drops vs BOW (driven more by genuinely shared words like 'sat'/'on').


Think of L2-normalization as converting raw "scores" into "percentages of the whole," so documents of different sizes can be compared fairly.

Imagine two recipes: one is a tiny snack with 3 ingredients, the other is a big feast with 30 ingredients. If you just compare raw ingredient amounts, the feast will look "bigger" or "more similar" to everything just because it has more stuff in it overall — not because the actual flavor profile matches.

L2-normalization fixes this by rescaling each document's whole set of numbers down (or up) so the *total "size"* of every document's number-list becomes the same standard amount. It doesn't change which words mattered more than others within the document — a word that was twice as important as another word stays twice as important. It just removes the effect of "this document happened to be longer or had bigger raw numbers."

That's why, in the TF-IDF example, "the" started out with a raw score of 2.446 (2 occurrences × its importance weight) but ended up printed as 0.578 — the whole document's numbers got scaled down together so the document's overall "size" matches a fixed standard, making it comparable to other documents regardless of their length.

The practical benefit: when we then measure how similar two documents are, we're comparing their *flavor profiles* (the relative emphasis on each word) rather than being misled by one document simply being longer and having bigger numbers across the board.

# 4. WORD2VEC (dense, static embeddings)

NOTE: A real Word2Vec needs millions of words to learn good vectors. This tutorial just shows the MECHANICS, not meaningful semantics.

In [10]:
from gensim.models import Word2Vec

tokenized = [doc.split() for doc in corpus]
w2v_model = Word2Vec(tokenized, vector_size=20, window=3, min_count=1, sg=1, seed=42, workers=1)

print(f"\nEmbedding dimension: {w2v_model.vector_size} (dense, vs {len(vocab)} for one-hot)")
cat_w2v = w2v_model.wv["cat"]
dog_w2v = w2v_model.wv["dog"]
king_w2v = w2v_model.wv["king"]

print(f"cosine(cat, dog)  = {cosine_similarity([cat_w2v], [dog_w2v])[0][0]:.3f}")
print(f"cosine(cat, king) = {cosine_similarity([cat_w2v], [king_w2v])[0][0]:.3f}")
print(">> On a real large corpus, cat/dog would be far closer than cat/king.")
print("   (The corpus here is too small to learn that reliably -- use pretrained")
print("    vectors like Google's GoogleNews-vectors or GloVe for a real demo.)")


Embedding dimension: 20 (dense, vs 15 for one-hot)
cosine(cat, dog)  = -0.254
cosine(cat, king) = -0.324
>> On a real large corpus, cat/dog would be far closer than cat/king.
   (The corpus here is too small to learn that reliably -- use pretrained
    vectors like Google's GoogleNews-vectors or GloVe for a real demo.)


**Embedding dimension: 20 (dense, vs 15 for one-hot)**

Each word now gets a vector of just 20 numbers, where every number can be any decimal value (not just 0s and 1s like one-hot). Even though one-hot vectors were length 15 here, embeddings stay a fixed, compact size (commonly 100–300 in real use) no matter how big the vocabulary grows — that's the "dense" advantage: more information packed into fewer numbers, instead of one mostly-empty slot per word.

**cosine(cat, dog) = -0.254, cosine(cat, king) = -0.324**

These are similarity scores between word vectors, like before. In a well-trained embedding, we'd expect "cat" and "dog" (both animals) to score noticeably higher than "cat" and "king" (unrelated). Here both scores are negative and fairly close to each other — meaning the model didn't learn any meaningful relationship at all.

**Why:** Word2Vec learns word meaning by observing patterns across *many* example sentences — it needs to see each word used in lots of different contexts to figure out which words behave similarly. The toy corpus here has only 4 short sentences, so each word appeared in just one or two contexts — nowhere near enough for the model to learn anything real. The numbers it produced are essentially noise from too little data, not a meaningful judgment that cat and king are more different than cat and dog.

**The takeaway**: this step is included to show *how* Word2Vec works mechanically (turning words into compact, trainable vectors), not to prove it works well. To actually see "cat" and "dog" land close together and "king" land farther away, you'd need a model trained on millions of real sentences — which is exactly what pretrained vectors like Google's GoogleNews-vectors or GloVe provide.

# 5. PRETRAINED CONTEXTUAL EMBEDDINGS (Sentence-BERT)

Note: REQUIRES INTERNET on first run (downloads model from Hugging Face, ~80MB).
Skip gracefully if offline so the rest of the demo still runs.

In [11]:
try:
    from sentence_transformers import SentenceTransformer

    model = SentenceTransformer("all-MiniLM-L6-v2")
    sbert_embeddings = model.encode(corpus)

    print(f"Embedding dimension: {sbert_embeddings.shape[1]}")
    sbert_sim = cosine_similarity(sbert_embeddings)
    print("\nDocument similarity matrix (Sentence-BERT):")
    print(np.round(sbert_sim, 2))
    print(">> D0 (cat/mat) and D1 (dog/rug) should now be rated highly similar")
    print("   because the MEANING is similar (pet resting on a surface),")
    print("   even though they share few exact words besides stopwords.")
    print(">> D3 (stock prices) should be clearly separated from the rest.")
    sbert_available = True
except Exception as e:
    print(f"[Skipped: could not download pretrained model -- {type(e).__name__}]")
    print("This step needs internet access on first run (downloads from")
    print("huggingface.co). Run this on Colab/local Jupyter with internet,")
    print("or pre-download the model beforehand.")
    sbert_available = False

c:\Users\matthieu.catteyfaye\Documents\NLP\.venv_nlp\Lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm
c:\Users\matthieu.catteyfaye\Documents\NLP\.venv_nlp\Lib\site-packages\huggingface_hub\file_download.py:143: UserWarning: `huggingface_hub` cache-system uses symlinks by default to efficiently store duplicated files but your machine does not support them in C:\Users\matthieu.catteyfaye\.cache\huggingface\hub\models--sentence-transformers--all-MiniLM-L6-v2. Caching files will still work but in a degraded version that might require more space on your disk. This warning can be disabled by setting the `HF_HUB_DISABLE_SYMLINKS_WARNING` environment variable. For more details, see https://huggingface.co/docs/huggingface_hub/how-to-cache#limitations.
To support symlinks on Windows, you either need to activate Developer Mod

Embedding dimension: 384

Document similarity matrix (Sentence-BERT):
[[ 1.    0.56  0.02  0.06]
 [ 0.56  1.   -0.    0.1 ]
 [ 0.02 -0.    1.    0.  ]
 [ 0.06  0.1   0.    1.  ]]
>> D0 (cat/mat) and D1 (dog/rug) should now be rated highly similar
   because the MEANING is similar (pet resting on a surface),
   even though they share few exact words besides stopwords.
>> D3 (stock prices) should be clearly separated from the rest.


This is the Sentence-BERT section — the pretrained, context-aware model — and it shows a fundamentally different kind of similarity than BOW or TF-IDF: similarity based on *meaning*, not on which exact words overlap.

**Embedding dimension: 384**

Each entire sentence (not each word) gets compressed into a single dense vector of 384 numbers, regardless of how many words the sentence contains. This is larger than Word2Vec's 20 dimensions because Sentence-BERT was pretrained on massive amounts of real text and needs more numbers to capture rich, context-dependent meaning.

**The similarity matrix**

| | D0 | D1 | D2 | D3 |
|---|---|---|---|---|
| **D0** | 1.00 | 0.56 | 0.02 | 0.06 |
| **D1** | 0.56 | 1.00 | -0.00 | 0.10 |
| **D2** | 0.02 | -0.00 | 1.00 | 0.00 |
| **D3** | 0.06 | 0.10 | 0.00 | 1.00 |

D0–D1 ("the cat sat on the mat" vs "the dog sat on the rug") score 0.56 — clearly the highest pair, even though earlier with TF-IDF this pair also scored highest mostly because of the shared word "sat"/"on"/"the". Here it's different: the model recognizes "cat...mat" and "dog...rug" describe the *same kind of situation* (a pet resting on a surface), even if you swapped "cat" for "dog" and "mat" for "rug" entirely.

D0–D2 and D1–D2 drop to nearly 0. "The king ruled the kingdom" is about governance, not pets — a completely different topic, so Sentence-BERT correctly rates it as unrelated, despite sharing the word "the" with D0 and D1.

D3 ("stock prices rose sharply today") also stays near 0 with everything else — finance news has nothing to do with pets or kingdoms.

**The key contrast with BOW/TF-IDF**: earlier methods judged similarity by counting shared words. Sentence-BERT judges similarity by understanding what the sentence is *about*. That's why D0 and D1 still come out as the most similar pair — but now for the right reason (similar meaning), rather than the previous, more superficial reason (shared stopwords and verbs).

# 6. CONTEXT-AWARENESS DEMO: same word, different meaning

In [12]:
if sbert_available:
    bank_sentences = [
        "I deposited money at the bank",
        "He sat on the river bank",
    ]
    bank_embeddings = model.encode(bank_sentences)
    bank_sim = cosine_similarity([bank_embeddings[0]], [bank_embeddings[1]])[0][0]

    print(f"Sentence 1: {bank_sentences[0]}")
    print(f"Sentence 2: {bank_sentences[1]}")
    print(f"\nSentence-BERT similarity of the two sentences: {bank_sim:.3f}")
    print(">> Even though both contain the word 'bank', BERT-style models give")
    print("   each sentence a different overall representation based on context.")
    print("   Word2Vec, by contrast, would assign 'bank' the SAME vector in both --")
    print("   it cannot tell financial bank from river bank.")
else:
    print("[Skipped -- see note above]")

print("\n" + "=" * 70)
print("SUMMARY TAKEAWAY")
print("=" * 70)
print("""
OHC      : no similarity at all between any two distinct words
BOW      : similarity driven by shared surface words (incl. stopwords)
TF-IDF   : similarity driven by shared DISTINCTIVE words
Word2Vec : similarity reflects learned semantic relationships (needs big corpus)
S-BERT   : similarity reflects sentence MEANING, context-aware, best for paraphrase/semantic search
""")

Sentence 1: I deposited money at the bank
Sentence 2: He sat on the river bank

Sentence-BERT similarity of the two sentences: 0.376
>> Even though both contain the word 'bank', BERT-style models give
   each sentence a different overall representation based on context.
   Word2Vec, by contrast, would assign 'bank' the SAME vector in both --
   it cannot tell financial bank from river bank.

SUMMARY TAKEAWAY

OHC      : no similarity at all between any two distinct words
BOW      : similarity driven by shared surface words (incl. stopwords)
TF-IDF   : similarity driven by shared DISTINCTIVE words
Word2Vec : similarity reflects learned semantic relationships (needs big corpus)
S-BERT   : similarity reflects sentence MEANING, context-aware, best for paraphrase/semantic search



# Implementation

TF-IDF applied to a simple NLP task: classifying short messages as
"spam" or "not spam" (ham).

Pipeline:
  1. Small labeled text dataset
  2. Convert text -> TF-IDF vectors
  3. Train a simple classifier (Logistic Regression) on those vectors
  4. Evaluate on held-out test messages
  5. Inspect which words the model learned are most "spammy"

This is intentionally small and dependency-light so the FULL pipeline
is visible in one file -- in real projects you'd use a much bigger
dataset, but the steps are identical.

In [13]:
import numpy as np
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.linear_model import LogisticRegression
from sklearn.model_selection import train_test_split
from sklearn.metrics import accuracy_score, classification_report

# ---------------------------------------------------------
# 1. DUMMY LABELED DATASET
# label: 1 = spam, 0 = not spam (ham)
# ---------------------------------------------------------
messages = [
    "win a free prize now click here",
    "congratulations you won a free lottery ticket",
    "claim your free cash prize today",
    "urgent click this link to win money",
    "you have been selected for a free gift card",
    "limited time offer claim your reward now",
    "act now to win a brand new car for free",
    "free entry to win cash, click immediately",

    "let's meet for lunch tomorrow at noon",
    "can you send me the report by friday",
    "the meeting has been moved to 3pm",
    "happy birthday hope you have a great day",
    "don't forget to pick up groceries on the way home",
    "the project deadline is next monday",
    "are we still on for coffee this weekend",
    "thanks for your help with the presentation",
]

labels = [1, 1, 1, 1, 1, 1, 1, 1,   # spam
          0, 0, 0, 0, 0, 0, 0, 0]   # ham

print("=" * 70)
print("DATASET")
print("=" * 70)
for msg, lab in zip(messages, labels):
    tag = "SPAM" if lab == 1 else "HAM "
    print(f"[{tag}] {msg}")

# ---------------------------------------------------------
# 2. TRAIN / TEST SPLIT
# Keep it stratified so both classes appear in train and test.
# ---------------------------------------------------------
X_train_text, X_test_text, y_train, y_test = train_test_split(
    messages, labels, test_size=0.25, random_state=42, stratify=labels
)

print("\n" + "=" * 70)
print(f"Train size: {len(X_train_text)}   Test size: {len(X_test_text)}")
print("=" * 70)

# ---------------------------------------------------------
# 3. TEXT -> TF-IDF VECTORS
# IMPORTANT: fit the vectorizer ONLY on training text, then use the
# same fitted vectorizer to transform the test text. This avoids
# "leaking" information from the test set into the vocabulary/IDF
# statistics -- a common real-world mistake.
# ---------------------------------------------------------
vectorizer = TfidfVectorizer(stop_words="english")

X_train_tfidf = vectorizer.fit_transform(X_train_text)
X_test_tfidf = vectorizer.transform(X_test_text)

print(f"\nVocabulary size (after removing stopwords): {len(vectorizer.get_feature_names_out())}")
print(f"Train matrix shape: {X_train_tfidf.shape}  (messages x vocab)")
print(f"Test matrix shape:  {X_test_tfidf.shape}")

# ---------------------------------------------------------
# 4. TRAIN A SIMPLE CLASSIFIER ON THE TF-IDF VECTORS
# Logistic Regression is a good first choice: fast, interpretable,
# and works well on sparse high-dimensional TF-IDF features.
# ---------------------------------------------------------
clf = LogisticRegression()
clf.fit(X_train_tfidf, y_train)

# ---------------------------------------------------------
# 5. EVALUATE ON THE TEST SET
# ---------------------------------------------------------
y_pred = clf.predict(X_test_tfidf)

print("\n" + "=" * 70)
print("TEST SET PREDICTIONS")
print("=" * 70)
for msg, true_label, pred_label in zip(X_test_text, y_test, y_pred):
    true_tag = "SPAM" if true_label == 1 else "HAM "
    pred_tag = "SPAM" if pred_label == 1 else "HAM "
    correct = "OK" if true_label == pred_label else "WRONG"
    print(f"[true={true_tag}] [pred={pred_tag}] [{correct}]  {msg}")

print(f"\nAccuracy: {accuracy_score(y_test, y_pred):.2f}")
print("\nDetailed report:")
print(classification_report(y_test, y_pred, target_names=["ham", "spam"], zero_division=0))

# ---------------------------------------------------------
# 6. INSPECT THE MODEL: which words push toward "spam"?
# Logistic Regression assigns one weight per TF-IDF feature (word).
# Larger positive weight -> pushes prediction toward spam (label 1).
# Larger negative weight -> pushes prediction toward ham (label 0).
# ---------------------------------------------------------
feature_names = vectorizer.get_feature_names_out()
weights = clf.coef_[0]

top_spam_idx = np.argsort(weights)[-8:][::-1]
top_ham_idx = np.argsort(weights)[:8]

print("\n" + "=" * 70)
print("TOP WORDS PUSHING TOWARD 'SPAM'")
print("=" * 70)
for idx in top_spam_idx:
    print(f"{feature_names[idx]:15s}  weight = {weights[idx]:+.3f}")

print("\n" + "=" * 70)
print("TOP WORDS PUSHING TOWARD 'HAM'")
print("=" * 70)
for idx in top_ham_idx:
    print(f"{feature_names[idx]:15s}  weight = {weights[idx]:+.3f}")

# ---------------------------------------------------------
# 7. TRY IT ON A BRAND NEW MESSAGE
# ---------------------------------------------------------
new_messages = [
    "free cash prize click now to claim",
    "can we reschedule our meeting to thursday",
]

new_tfidf = vectorizer.transform(new_messages)
new_pred = clf.predict(new_tfidf)
new_proba = clf.predict_proba(new_tfidf)

print("\n" + "=" * 70)
print("PREDICTIONS ON NEW, UNSEEN MESSAGES")
print("=" * 70)
for msg, pred, proba in zip(new_messages, new_pred, new_proba):
    tag = "SPAM" if pred == 1 else "HAM"
    print(f"[{tag}] (P(spam)={proba[1]:.2f})  {msg}")

DATASET
[SPAM] win a free prize now click here
[SPAM] congratulations you won a free lottery ticket
[SPAM] claim your free cash prize today
[SPAM] urgent click this link to win money
[SPAM] you have been selected for a free gift card
[SPAM] limited time offer claim your reward now
[SPAM] act now to win a brand new car for free
[SPAM] free entry to win cash, click immediately
[HAM ] let's meet for lunch tomorrow at noon
[HAM ] can you send me the report by friday
[HAM ] the meeting has been moved to 3pm
[HAM ] happy birthday hope you have a great day
[HAM ] don't forget to pick up groceries on the way home
[HAM ] the project deadline is next monday
[HAM ] are we still on for coffee this weekend
[HAM ] thanks for your help with the presentation

Train size: 12   Test size: 4

Vocabulary size (after removing stopwords): 42
Train matrix shape: (12, 42)  (messages x vocab)
Test matrix shape:  (4, 42)

TEST SET PREDICTIONS
[true=HAM ] [pred=HAM ] [OK]  happy birthday hope you have a great da


KEY TAKEAWAYS
---

1. TF-IDF turns each message into a fixed-length numeric vector,
   so a standard ML classifier (Logistic Regression here) can use it.
2. Fit the vectorizer on TRAIN data only, then transform test/new data
   with the same fitted vectorizer -- never re-fit on test data.
3. The classifier's learned weights show which words drive each
   prediction -- a simple form of model interpretability that dense
   embeddings (Word2Vec/BERT) don't give you as directly.
4. This same TF-IDF + classifier pattern generalizes to many tasks:
   topic classification, sentiment analysis, document categorization.